In [1]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from functools import reduce
import joblib
from typing import Dict
import gc

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector
from src.utils.data_structs import TripletCreator, NodeCreator, Relation, NODES_TYPES_MAP, RELATIONS_TYPES_MAP
from src.llm_agent.agent_model import SYSTEM_PROMPT

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

DATASET_PATH = '../data/Augment_DiaASQ.json'
LOAD_EXTRACTED_TRIPLETS_FILE = ''
SAVE_GRAPH_DB_NAME = 'diaasq2'
gc.collect()

20

### Update

In [32]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://localhost:7687", user="neo4j", pwd="password", db_name=SAVE_GRAPH_DB_NAME),
    embeddings_db=EmbeddingsDatabaseConnection(EmbeddingsDatabaseConnectionConfig(
        nodes_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_nodes/v10/densedb', 'vectorized_nodes', is_exist=False, need_to_clear=False
        ),
        triplets_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_triplets/v6/densedb', 'vectorized_triplets', is_exist=False, need_to_clear=False
        )
    ))
)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with MEAN pooling.


In [4]:
extracted_triplets = joblib.loads(LOAD_EXTRACTED_TRIPLETS_FILE)
print(len(extracted_triplets))

3483


In [16]:
extracted_triplets = reduce(lambda acc, v: acc + v, extracted_triplets, [])
print(len(extracted_triplets))

283268


In [34]:
kg_model.graph_db.create_triplets(extracted_triplets)

  0%|          | 0/283268 [00:00<?, ?it/s]Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: object)} {position: line: 1, column: 13, offset: 12} for query: 'MATCH (subj:object) WHERE subj.name = "gregory" RETURN elementID(subj) as id'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your

In [ ]:
kg_model.graph_db.close()